In [1]:
from code_prompt.evaluation import evaluate_directory_zero_shot
import json

datasets = [
    "agnews",
    "cola",
    "iris",
    "mrpc",
    "mscinli",
    "scierc",
    "semeval",
    "sst",
    "xnli",
]
for dataset in ["sst"]:
    result = evaluate_directory_zero_shot(f"outputs/prompt")
    # save as json
    with open(f"zeroshot-{dataset}-evaluation.json", "w") as f:
        json.dump(result, f, indent=4)

In [4]:
from code_prompt.create_excel import create_excel

datasets = ["agnews", "cola", "iris", "mrpc", "mscinli", "scierc", "semeval", "xnli"]
for dataset in ["semeval"]:
    create_excel(dataset, score="avg_redundancy")
    create_excel(dataset, score="f1_micro")

In [4]:
from code_prompt.evaluation import evaluate_directory_few_shot
from code_prompt.create_excel import create_excel
import json

datasets = [
    "sst",
    "agnews",
    "cola",
    "semeval",
    "scierc",
    "xnli",
    "mscinli",
    "mrpc",
    "iris",
    "hcc",
]
for dataset in ["mscinli"]:
    result = evaluate_directory_few_shot(f"outputs/fewshot/{dataset}")
    # save as json
    with open(f"eval/fewshot-{dataset}-evaluation.json", "w") as f:
        json.dump(result, f, indent=4)
    create_excel(dataset, score="avg_redundancy", scenario="fewshot")
    create_excel(dataset, score="f1_macro", scenario="fewshot")

In [2]:
from code_prompt.create_excel import create_excel

datasets = ["agnews", "cola", "iris", "mrpc", "mscinli", "scierc", "semeval", "xnli"]
for dataset in ["sst", "agnews"]:
    create_excel(dataset, score="redundancy", scenario="fewshot")
    create_excel(dataset, score="accuracy", scenario="fewshot")

In [3]:
from code_prompt.evaluation import evaluate_directory_programming_language
import json


for dataset in ["xnli_multi"]:
    result = evaluate_directory_programming_language(f"outputs/{dataset}")
    # save as json
    with open(f"pl-{dataset}-evaluation.json", "w") as f:
        json.dump(result, f, indent=4)

In [9]:
# read json as list of dicts
import json

with open("pl-xnli_multi-evaluation.json", "r") as f:
    records = json.load(f)

def get_model(filename):
    if "CodeLlama" in filename:
        return "CodeLlama"
    elif "Qwen" in filename:
        return "Qwen"
    elif "codegemma" in filename:
        return "CodeGemma"
    elif "gpt" in filename:
        return "GPT-3.5"
    elif "deepseek" in filename:
        return "DeepSeek"

for record in records:
    record["language"] = record["filename"].split("_")[2]
    record["model"] = get_model(record["filename"])


In [13]:
"""Given a list of dict with each element looking like
{'filename': 'xnli_multi_ar_CodeLlama-13b-hf_0_shot.json',
  'dataset': 'xnli',
  'model': 'CodeLlama',
  'shot': 0,
  'seed': None,
  'language': 'ar',
  'total': 5010,
  'correct': 2289,
  'accuracy': 0.4568862275449102,
  'avg_redundancy': 0.6712880959240555},

generate an excel file with "language" as columns and "model" as rows, and accuracy as values
language is sorted alphabetically, model is sorted by "DeepSeek", "GPT-3.5", "Qwen", "CodeGemma", "CodeLlama"
accuracy is rounded to 3 digit percentage, such as 45.7
the last row is avg + std of each column, such as "45.7 ± 3.2"
"""

import pandas as pd
df = pd.DataFrame(records)
df = df.pivot(index="model", columns="language", values="accuracy")
df = df[sorted(df.columns)]
df = df.reindex(["DeepSeek", "GPT-3.5", "Qwen", "CodeGemma", "CodeLlama"])
df = df.applymap(lambda x: f"{x*100:.1f}" if pd.notnull(x) else x)
df.loc["avg ± std"] = df.apply(lambda x: f"{x.astype(float).mean():.1f} ± {x.astype(float).std():.1f}")
df.to_excel("pl-xnli_multi-evaluation.xlsx")

/tmp/ipykernel_15868/1928031094.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: f"{x*100:.1f}" if pd.notnull(x) else x)


68.5 ± 10.4
